# LangGraph example: a three-layer number pipeline

Type a number. One agent checks it is a whole number from 1 to 10, three agents each own a range, three more add facts, and a last one confirms the number. Shaped like the incident-triage graph, with no Azure and no LLM.

**To run it:** Python 3.12 and `pip install langgraph`.

## The graph

Same shape as the incident-triage graph, with numbers instead of incidents.

```
                      +--> low  (1-3) --+      +--> even_odd --+
 START -> validate ---+--> mid  (4-7) --+------+--> prime    --+--> finalize -> END
             |        +--> high (8-10) -+      +--> size     --+
             +-- not a number 1-10 --> reject -> END
```

| Layer | Nodes | Like in the project | LangGraph feature |
|---|---|---|---|
| 1 | `validate` | Triage supervisor | conditional edge (valid or reject) |
| 2 | `low`, `mid`, `high` | Logs, Metrics, Change agents | fan-out, reducer on `findings` |
| 3 | `even_odd`, `prime`, `size` | a second wave of specialists | fan-in (wait for all of layer 2), fan-out again |
| 4 | `finalize` | Postmortem | fan-in, returns the final answer |

Each node follows the project's pattern: read state, say what it is doing (`trace`), do its job, return only its own piece.

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


def merge_findings(a: dict, b: dict) -> dict:       # reducer, same idea as the project's _merge
    return {**a, **b}


class Pipeline(TypedDict, total=False):
    raw: str                                              # what the user typed
    number: int                                           # set by validate
    error: str                                            # set by validate when the input is bad
    findings: Annotated[dict, merge_findings]             # layers 2 and 3 all write here
    trace: Annotated[list, operator.add]                  # every node appends a line (like em.say)
    final: int                                            # set by finalize


def say(node: str, text: str) -> dict:
    return {"trace": [f"{node:9} {text}"]}


# ---- layer 1: the supervisor
def validate(state: Pipeline) -> dict:
    raw = state["raw"].strip()
    if not raw.lstrip("-").isdigit():
        return {"error": f"'{raw}' is not a whole number", **say("validate", "rejected: not a number")}
    n = int(raw)
    if not 1 <= n <= 10:
        return {"error": f"{n} is outside 1-10", **say("validate", "rejected: out of range")}
    return {"number": n, **say("validate", f"{n} is valid, sending to the range agents")}


LAYER2 = ["low", "mid", "high"]
LAYER3 = ["even_odd", "prime", "size"]


def route_after_validate(state: Pipeline):
    return "reject" if state.get("error") else LAYER2     # a list means: run all of these in parallel


def reject(state: Pipeline) -> dict:
    return say("reject", state["error"])


# ---- layer 2: three range agents, run in parallel, each checks its own range
def range_agent(name: str, low: int, high: int):
    def node(state: Pipeline) -> dict:
        mine = low <= state["number"] <= high
        text = "my range, claiming it" if mine else "not mine"
        return {"findings": {name: mine}, **say(name, f"{low}-{high}: {text}")}
    node.__name__ = name
    return node


# ---- layer 3: three more agents, run in parallel after ALL of layer 2 finished
def even_odd(state: Pipeline) -> dict:
    kind = "even" if state["number"] % 2 == 0 else "odd"
    return {"findings": {"parity": kind}, **say("even_odd", kind)}


def prime(state: Pipeline) -> dict:
    n = state["number"]
    is_prime = n > 1 and all(n % d for d in range(2, n))
    return {"findings": {"prime": is_prime}, **say("prime", "prime" if is_prime else "not prime")}


def size(state: Pipeline) -> dict:
    n = state["number"]
    label = "small" if n <= 3 else "medium" if n <= 7 else "large"
    return {"findings": {"size": label}, **say("size", label)}


# ---- layer 4: finalize, waits for layer 3, returns the number
def finalize(state: Pipeline) -> dict:
    owner = next(k for k in ("low", "mid", "high") if state["findings"][k])
    return {"final": state["number"], **say("finalize", f"{state['number']} confirmed by '{owner}'")}


In [2]:
g = StateGraph(Pipeline)
g.add_node("validate", validate)
g.add_node("reject", reject)
g.add_node("low", range_agent("low", 1, 3))
g.add_node("mid", range_agent("mid", 4, 7))
g.add_node("high", range_agent("high", 8, 10))
for fn in (even_odd, prime, size, finalize):
    g.add_node(fn.__name__, fn)

g.add_edge(START, "validate")
g.add_conditional_edges("validate", route_after_validate, ["reject", *LAYER2])
g.add_edge("reject", END)
for agent in LAYER3:
    g.add_edge(LAYER2, agent)
g.add_edge(LAYER3, "finalize")
g.add_edge("finalize", END)
pipeline = g.compile()


def run(text: str):
    out = pipeline.invoke({"raw": text})
    print(f"input {text!r}")
    for line in out["trace"]:
        print("   ", line)
    print("    final:", out.get("final"), "| error:", out.get("error"), "\n")


run("6")          # valid: mid claims it
run("2")          # valid: low claims it
run("11")         # out of range: rejected in layer 1, layers 2 and 3 never run
run("abc")        # not a number: rejected in layer 1

input '6'
    validate  6 is valid, sending to the range agents
    high      8-10: not mine
    low       1-3: not mine
    mid       4-7: my range, claiming it
    even_odd  even
    prime     not prime
    size      medium
    finalize  6 confirmed by 'mid'
    final: 6 | error: None 

input '2'
    validate  2 is valid, sending to the range agents
    high      8-10: not mine
    low       1-3: my range, claiming it
    mid       4-7: not mine
    even_odd  even
    prime     prime
    size      small
    finalize  2 confirmed by 'low'
    final: 2 | error: None 

input '11'
    validate  rejected: out of range
    reject    11 is outside 1-10
    final: None | error: 11 is outside 1-10 

input 'abc'
    validate  rejected: not a number
    reject    'abc' is not a whole number
    final: None | error: 'abc' is not a whole number 



**What to notice**

| Observation | Why |
|---|---|
| `validate` always prints first | it is the only node after `START` |
| `low`, `mid`, `high` all run, even when two say "not mine" | fan-out runs every branch; each decides for itself |
| Layer 3 starts only after all three range agents | a list as the source of `add_edge` means "wait for all" |
| `finalize` prints last | it waits for all of layer 3 |
| Bad input skips layers 2 to 4 | the conditional edge sends it to `reject` |
| `findings` holds keys from six different nodes | the reducer merges parallel writes (section 2) |
| Order inside a layer may differ between runs | parallel nodes have no fixed order |

## Where to look next in the project

| Concept | File |
|---|---|
| State, reducer, nodes, edges, parallel, conditional edges | `backend/src/triage/graph.py` |
| Agent pattern (read, announce, act, report) | `backend/src/triage/agents.py` |